# ハンズオン③ HuggingFace で LLM 推論パラメータを探索する

**対応セクション**: 1-5「推論パラメータ」  
**推奨所要時間**: 約 30 分  
**使用ライブラリ**: transformers, torch, matplotlib

---

## このノートブックの目標

1. HuggingFace `transformers` で LLM をロードし、テキストを生成する
2. `temperature` が出力の多様性・確実性に与える影響を体感する
3. `top_p` によるサンプリング絞り込みの効果を確認する
4. 同一パラメータで複数回生成し、出力のばらつきを定量的に可視化する

> **モデルについて**: `meta-llama/Llama-3.2-1B-Instruct`（約 2.5 GB）を使用します。  
> 初回実行時はダウンロードが発生します。共有ストレージ（`/data/shared/models`）に  
> キャッシュがある場合は自動的にそちらを使用します。

## セットアップ

In [ ]:
# 実行時間: 数秒
import os
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM

# 共有ストレージがあればキャッシュ先として使用
if os.path.exists('/data/shared/models'):
    os.environ['HF_HOME'] = '/data/shared/models'
    print('キャッシュ先: /data/shared/models')
else:
    print('キャッシュ先: デフォルト (~/.cache/huggingface)')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')
print(f'PyTorch バージョン: {torch.__version__}')

# 再現性のためシードを固定（サンプリング実験ではあえて外す場合もある）
torch.manual_seed(42)

---

## Step 1: モデルとトークナイザーの読み込み

今回は軽量な指示チューニング済みモデル `Llama-3.2-1B-Instruct` を使用します。  
`float16` でロードすることでメモリ使用量を半減させます。

In [ ]:
# 実行時間: 初回は数分（ダウンロード）、2回目以降は約30秒

MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

print(f'モデルをロード中: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # メモリ節約
    device_map='auto',           # 利用可能なデバイスに自動配置
)
model.eval()  # 推論モード（dropout 等を無効化）

# パラメータ数の確認
total_params = sum(p.numel() for p in model.parameters())
print(f'\nロード完了')
print(f'総パラメータ数: {total_params / 1e9:.2f}B')
print(f'語彙サイズ: {tokenizer.vocab_size:,}')

---

## Step 2: 基本推論

まず最もシンプルな生成を確認します。  
Instruct モデルには「システムプロンプト + ユーザーメッセージ」の形式でプロンプトを渡します。

In [ ]:
# 実行時間: GPU で約5秒、CPU で約1〜2分

def build_prompt(user_message: str, system: str = 'You are a helpful assistant.') -> str:
    """Llama-3 の chat template に合わせたプロンプトを組み立てる"""
    messages = [
        {'role': 'system',    'content': system},
        {'role': 'user',      'content': user_message},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_text(
    prompt: str,
    max_new_tokens: int = 150,
    do_sample: bool = True,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
) -> str:
    """プロンプトを受け取り、生成テキスト（プロンプト部分を除く）を返す"""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id,
        )

    # プロンプト部分を除いた生成トークンだけデコード
    generated_ids = outputs[0][input_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


# 動作確認
question = '深層学習を一言で説明してください。'
prompt   = build_prompt(question)
response = generate_text(prompt)

print(f'質問: {question}')
print(f'回答: {response}')

---

## Step 3: Temperature の実験

同一プロンプトに対して 3 種類の temperature で生成し、出力の傾向を比較します。

| temperature | 期待される傾向 |
|---|---|
| `0.1` | 確実・再現性高・同じ表現を繰り返しやすい |
| `0.7` | バランス型（多くの場面でのデフォルト） |
| `1.3` | 多様・創造的・ときに意外な表現が出る |

In [ ]:
# 実行時間: GPU で約15秒、CPU で約5分

prompt_temp = build_prompt('AIの未来について、自由に語ってください。')
temperatures = [0.1, 0.7, 1.3]
colors = ['#0D4A38', '#7C5C00', '#991B1B']

temp_results = {}

for t in temperatures:
    torch.manual_seed(42)  # 比較のためシード固定
    response = generate_text(
        prompt_temp,
        max_new_tokens=120,
        temperature=t,
        top_p=0.9,
    )
    temp_results[t] = response
    print(f'\n{'='*60}')
    print(f'temperature = {t}')
    print('='*60)
    print(response)

### Temperature によるトークン多様性の定量比較

各出力に含まれるユニークトークンの比率（type-token ratio）で多様性を数値化します。

$$\text{TTR} = \frac{\text{ユニークトークン数}}{\text{総トークン数}}$$

In [ ]:
# 実行時間: 数秒

def type_token_ratio(text: str) -> float:
    """テキストのトークン多様性（TTR）を計算する"""
    token_ids = tokenizer.encode(text)
    return len(set(token_ids)) / len(token_ids) if token_ids else 0.0


ttrs = {t: type_token_ratio(resp) for t, resp in temp_results.items()}

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(
    [str(t) for t in temperatures],
    list(ttrs.values()),
    color=colors, width=0.5
)
for bar, val in zip(bars, ttrs.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('temperature')
ax.set_ylabel('Type-Token Ratio（高いほど多様）')
ax.set_title('Temperature とトークン多様性の関係')
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

for t, ttr in ttrs.items():
    print(f'temperature={t:.1f} | TTR={ttr:.3f}')

---

## Step 4: Top-p の実験

Temperature を固定（`0.9`）し、`top_p` だけを変えて出力の変化を確認します。  
Top-p が小さいほど候補が絞られ、確実性が上がります。

In [ ]:
# 実行時間: GPU で約15秒、CPU で約5分

prompt_topp = build_prompt('料理のレシピを1つ考えてください。')
top_ps = [0.5, 0.9, 1.0]

topp_results = {}

for p in top_ps:
    torch.manual_seed(42)
    response = generate_text(
        prompt_topp,
        max_new_tokens=120,
        temperature=0.9,
        top_p=p,
    )
    topp_results[p] = response
    print(f'\n{'='*60}')
    print(f'top_p = {p}')
    print('='*60)
    print(response)

---

## Step 5: 同一パラメータで複数回生成し、ばらつきを可視化

同じプロンプト・同じパラメータで 5 回生成し、  
出力の「ばらつき」をトークン共通率で可視化します。

- **低 temperature**: どの回も似た出力になる → 共通率が高い  
- **高 temperature**: 毎回異なる出力になる → 共通率が低い

In [ ]:
# 実行時間: GPU で約40秒（3条件 × 5回生成）、CPU で約15分

def token_overlap(text_a: str, text_b: str) -> float:
    """2つのテキスト間のトークン集合の Jaccard 類似度を計算する"""
    set_a = set(tokenizer.encode(text_a))
    set_b = set(tokenizer.encode(text_b))
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


N_TRIALS = 5
prompt_var = build_prompt('今日の気分を一文で表現してください。')
test_temps = [0.1, 0.7, 1.3]
colors_var  = ['#0D4A38', '#7C5C00', '#991B1B']

# 各 temperature で N_TRIALS 回生成
all_outputs = {}
for t in test_temps:
    outputs = []
    for i in range(N_TRIALS):
        # シードを変えて毎回異なるサンプルを得る
        torch.manual_seed(i)
        resp = generate_text(prompt_var, max_new_tokens=60, temperature=t, top_p=0.9)
        outputs.append(resp)
    all_outputs[t] = outputs
    print(f'temperature={t}: {N_TRIALS}回生成完了')

# 全ペアの Jaccard 類似度を計算し、temperature ごとの平均を求める
from itertools import combinations

avg_overlaps = {}
for t, outputs in all_outputs.items():
    pairs = list(combinations(outputs, 2))
    avg_overlaps[t] = sum(token_overlap(a, b) for a, b in pairs) / len(pairs)

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左: 平均 Jaccard 類似度（高いほど出力が似ている＝低多様性）
axes[0].bar(
    [str(t) for t in test_temps],
    list(avg_overlaps.values()),
    color=colors_var, width=0.5
)
for i, val in enumerate(avg_overlaps.values()):
    axes[0].text(i, val + 0.005, f'{val:.3f}', ha='center', fontsize=10)
axes[0].set_xlabel('temperature')
axes[0].set_ylabel('平均 Jaccard 類似度（低いほど多様）')
axes[0].set_title('5回生成の平均ばらつき')
axes[0].set_ylim(0, 1.0)

# 右: 各回の出力テキスト（temperature=0.7 の例）
axes[1].axis('off')
t_show = 0.7
text_lines = [f'temperature = {t_show}　各回の出力:\n']
for i, resp in enumerate(all_outputs[t_show]):
    short = resp[:60] + ('…' if len(resp) > 60 else '')
    text_lines.append(f'[{i+1}] {short}')
axes[1].text(0.02, 0.95, '\n'.join(text_lines),
             transform=axes[1].transAxes, fontsize=8,
             va='top', ha='left', wrap=True,
             bbox=dict(boxstyle='round', facecolor='#F0EDE6', alpha=0.8))

plt.tight_layout()
plt.show()

---

## 実験: repetition_penalty の効果

`repetition_penalty` を変えると、同じ表現の繰り返しがどう変わるか確認します。  
特に長い出力を生成させたときに効果が顕著に現れます。

In [ ]:
# 実行時間: GPU で約15秒、CPU で約5分

prompt_rep = build_prompt('春について、詩のように語ってください。')
rep_penalties = [1.0, 1.2, 1.5]

for r in rep_penalties:
    torch.manual_seed(42)
    response = generate_text(
        prompt_rep,
        max_new_tokens=150,
        temperature=0.9,
        top_p=0.92,
        repetition_penalty=r,
    )
    # 繰り返しトークンの検出（簡易版）
    token_ids  = tokenizer.encode(response)
    bigrams    = list(zip(token_ids, token_ids[1:]))
    dup_ratio  = 1 - len(set(bigrams)) / max(len(bigrams), 1)

    print(f'\n{'='*60}')
    print(f'repetition_penalty = {r}  |  2-gram 重複率: {dup_ratio:.3f}')
    print('='*60)
    print(response)

---

## まとめ

このノートブックで体感したこと:

1. **`temperature`** が低いほど出力は確実・一貫し、高いほど多様・創造的になる
2. **`top_p`** は累積確率で候補を絞り、低確率の「外れ値トークン」を排除する
3. 同じパラメータでも毎回異なる出力が生成される——サンプリングの確率的な性質
4. **`repetition_penalty`** は繰り返しを抑制するが、強くしすぎると文が不自然になる

次の章では、この事前学習済みモデルを特定のタスクに適応させる  
**ファインチューニング（SFT）** と **RAG（検索拡張生成）** を実装します。